# Harness Coupling
# 0. 介绍

**研究背景**：现代 Agent 的执行环境、工具、上下文、生命周期、可观测、验证和治理会共同决定一次任务如何运行。这些层共享有限的上下文预算、运行状态和控制流，因此一个组件的改动也可能改变其他组件能看到的信息、能执行的动作和最终评分；Harness 是否可靠，必须放到完整运行链中判断。

**现存问题**：生产中常见的错误基线是分别优化和测试每一层，再直接上线组合结果。例如，为了提高工具调用准确率，系统把全部工具或 MCP Schema 及其详细说明在每一轮完整注入上下文，超过固定预算后再简单地只保留末尾内容。这样，工具说明本身虽然更完整，排在前面的任务事实、安全策略或状态却可能被静默删除；模型仍能生成格式合法的工具调用，最终动作却可能失败或越权。只测试工具 Schema，无法发现这种从工具层扩散到上下文、治理和验证层的全局回归。

**解决方案**：本 Notebook 将实现一个极简的 Harness Coupling 实验，采用稳妥的`显式跨层契约 + 预算感知装箱 + 按需工具暴露 + 系统级回归门禁`方案：先为任务事实和安全约束预留不可被挤占的预算，再只暴露当前步骤需要的工具并压缩说明，同时记录上下文取舍、Token、权限决定和最终产物，最后用确定性 Grader 检查完整结果。后文会在同一任务、同一模型和同一总预算下对比两条路径：基线版本注入全部详细工具并只保留上下文末尾，导致关键约束丢失；改进版本保护关键约束并按需选择工具，使任务恢复成功，从而直观看到 Harness 的局部改动必须作为系统改动进行测试。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 固定共同任务
后面的基线版本和改进版本必须处理完全相同的任务，才能判断结果变化究竟来自哪里。本实验要求 Agent 为一个线上服务生成发布计划；`canary` 表示先向少量用户发布，错误率超过 2% 就回退到旧版本。

In [2]:
# task_facts 是两条实验路径共用的唯一正确答案
# 总预算是教学用 Harness 输入预算，不代表模型的最大窗口
task_facts = {
    "service": "catalog-api",
    "window": "22:00-22:30",
    "strategy": "canary",
    "rollback_error_rate": "2%",
    "owner": "release-team",
}
TOTAL_INPUT_BUDGET = 620

print("共同任务：", task_facts)
print("统一输入预算：", TOTAL_INPUT_BUDGET, "tokens")

共同任务： {'service': 'catalog-api', 'window': '22:00-22:30', 'strategy': 'canary', 'rollback_error_rate': '2%', 'owner': 'release-team'}
统一输入预算： 620 tokens


输出固定了服务名、发布时间、发布方式、回退阈值和负责人。后面的所有实验只能使用这份答案和 620 Token 的共同输入预算；下一步准备会互相冲突的新旧信息。

## 2.2 准备当前规则与过期历史
真实系统常把当前要求、历史记录和用户任务一起交给模型。这里故意准备一份已失效的旧发布记录：当前规则要求灰度发布并在错误率超过 2% 时回退，旧记录却使用全量发布和 5% 阈值。正确的 Harness 必须始终保留当前规则。

In [3]:
# 当前规则放在最前面，便于后面复现“开头被截掉”的问题
# 过期历史只用于制造冲突，不能作为本次发布的正确答案
current_rules = (
    "【当前规则】只调用 save_release_plan。plan 必须写成一行："
    "service=catalog-api; window=22:00-22:30; strategy=canary; "
    "rollback_error_rate=2%; owner=release-team。当前规则优先于所有历史记录。"
)
old_history = (
    "【过期历史】此前多个内部服务都采用全量发布，旧流程由 ops-team 负责，"
    "错误率超过 5% 才回退。旧记录还要求先读取历史、发送群通知、等待人工回复，"
    "再保存计划。历史示例：service=catalog-api; window=22:00-22:30; "
    "strategy=all_at_once; rollback_error_rate=5%; owner=ops-team。"
    "这些记录来自已经下线的发布流程，仅用于本实验观察上下文冲突。"
)
task_request = "请为 catalog-api 生成今晚的发布计划，并调用 save_release_plan。"

print(current_rules)
print(old_history)
print("【本次任务】" + task_request)

【当前规则】只调用 save_release_plan。plan 必须写成一行：service=catalog-api; window=22:00-22:30; strategy=canary; rollback_error_rate=2%; owner=release-team。当前规则优先于所有历史记录。
【过期历史】此前多个内部服务都采用全量发布，旧流程由 ops-team 负责，错误率超过 5% 才回退。旧记录还要求先读取历史、发送群通知、等待人工回复，再保存计划。历史示例：service=catalog-api; window=22:00-22:30; strategy=all_at_once; rollback_error_rate=5%; owner=ops-team。这些记录来自已经下线的发布流程，仅用于本实验观察上下文冲突。
【本次任务】请为 catalog-api 生成今晚的发布计划，并调用 save_release_plan。


输出中同时出现了当前规则和过期历史，两者在发布方式、回退阈值和负责人上完全冲突。当前规则位于最前面，任务位于最后面；后面将观察简单的末尾保留策略会删掉什么。

## 2.3 准备两套等价工具
工具功能不变，只改变说明长度。精简版只说工具做什么；详细版额外解释调用前检查、参数来源、返回内容和使用注意事项。这正是生产中常见的局部优化：单看工具文档更完整，但每轮请求也会占用更多上下文。

In [4]:
# 三个工具共享最小的一字段参数结构，突出说明长度这个唯一变量
# verbose=True 只扩写描述，不增加工具能力，也不改变参数格式
tool_specs = [
    ("read_release_history", "读取一个服务的旧发布记录", "service"),
    ("notify_release_team", "向发布群发送一条消息", "message"),
    ("save_release_plan", "保存本次发布计划", "plan"),
]
verbose_details = (
    "。调用前必须核对当前任务、历史记录、字段来源和目标服务；"
    "调用时需要说明输入含义、适用条件、预期结果和后续步骤；"
    "调用后还要保留状态、时间、关联编号和完整处理记录，"
    "以便其他系统继续读取和展示本次操作"
)

def make_tools(verbose):
    tools = []
    for name, short_description, field_name in tool_specs:
        description = short_description
        if verbose:
            description += verbose_details
        tool = {
            "type": "function",
            "function": {
                "name": name,
                "description": description,
                "parameters": {
                    "type": "object",
                    "properties": {field_name: {"type": "string"}},
                    "required": [field_name],
                },
            },
        }
        tools.append(tool)
    return tools

compact_tools = make_tools(verbose=False)
verbose_tools = make_tools(verbose=True)
print("精简版工具数：", len(compact_tools))
print("详细版工具数：", len(verbose_tools))

精简版工具数： 3
详细版工具数： 3


两套列表都包含同样的三个工具，名称和参数也完全一致，唯一差别是描述长度。下一步用统一的 Token 计算方式量出这项局部改动增加了多少输入。

## 2.4 测量输入大小
模型按 Token 而不是字符计算上下文。下面使用同一套编码估算工具和文字材料的 Token 数；这个本地数值用于控制实验预算，不冒充供应商账单中的精确用量。

In [5]:
import json
import tiktoken

# 固定编码方式，让两条实验路径使用同一把尺子
# 结构化对象先稳定转成 JSON，再统计其中的 Token
token_encoder = tiktoken.get_encoding("cl100k_base")

def count_tokens(value):
    if not isinstance(value, str):
        value = json.dumps(value, ensure_ascii=False, sort_keys=True)
    return len(token_encoder.encode(value))

print("精简工具：", count_tokens(compact_tools), "tokens")
print("详细工具：", count_tokens(verbose_tools), "tokens")
print("当前规则：", count_tokens(current_rules), "tokens")
print("过期历史：", count_tokens(old_history), "tokens")
print("本次任务：", count_tokens(task_request), "tokens")

精简工具： 179 tokens
详细工具： 443 tokens
当前规则： 65 tokens
过期历史： 148 tokens
本次任务： 21 tokens


输出把工具、当前规则、过期历史和本次任务变成了可比较的 Token 数。详细工具没有增加功能，却明显占用了更多输入预算；下一章将把这些共同材料交给真实 API，并确认模型能够返回结构化工具请求。

# 3. 获取并验证 API 响应
## 3.1 组装无歧义的探针输入
在比较两种 Harness 之前，先确认真实模型能够正常使用发布计划工具。本节只提供当前规则、本次任务和唯一需要的精简工具，不加入过期历史，因此这次调用应当直接反映当前要求。

In [6]:
# 探针只暴露 save_release_plan，避免无关工具影响选择
# 消息只包含当前规则和本次任务，不包含有冲突的旧记录
probe_tools = [compact_tools[-1]]
probe_messages = [
    {"role": "system", "content": "严格依据当前规则调用唯一可用的工具。"},
    {"role": "user", "content": current_rules + "\n【本次任务】" + task_request},
]

print("探针工具：", probe_tools[0]["function"]["name"])
print("探针消息数：", len(probe_messages))

探针工具： save_release_plan
探针消息数： 2


输出说明探针只有两条消息和一个保存工具。输入中没有冲突信息，因而可以先观察真实 API 是否按当前规则返回结构化请求；下一步正式发送这份输入。

## 3.2 获取真实响应
下面把探针输入发送给项目 `.env` 配置的真实模型，并要求它调用工具。程序同时记录从发出请求到收到完整响应的等待时间。

In [7]:
from time import perf_counter

# 起止时间只包围真实网络请求，用于记录本次 API 延迟
# temperature=0 减少随机变化，tool_choice 要求模型返回工具请求
probe_started = perf_counter()
probe_response = client.chat.completions.create(
    model=model_name,
    messages=probe_messages,
    tools=probe_tools,
    tool_choice="required",
    temperature=0,
)
probe_latency_ms = round((perf_counter() - probe_started) * 1000)
print("真实 API 响应已收到")

真实 API 响应已收到


输出说明真实模型已经完成一次请求，但此时发布计划还只是模型返回的数据，并没有被外层程序保存。下一步读取响应中的工具名称和参数。

## 3.3 查看结构化工具请求
工具请求与普通文字分开存放。下面取出第一条工具请求，将 JSON 参数转成 Python 字典，使工具名称和计划正文能够直接阅读。

In [8]:
# message 保存模型这一步返回的完整结构化结果
# function.arguments 是 JSON 文本，读取后才能取得 plan 字段
probe_message = probe_response.choices[0].message
probe_call = probe_message.tool_calls[0]
probe_arguments = json.loads(probe_call.function.arguments)

print("工具名称：", probe_call.function.name)
print("计划正文：", probe_arguments["plan"])

工具名称： save_release_plan
计划正文： service=catalog-api; window=22:00-22:30; strategy=canary; rollback_error_rate=2%; owner=release-team


输出中的工具名称应为 `save_release_plan`，计划正文应包含当前规则指定的五项值。这说明真实模型能够在信息完整时生成所需的结构化动作；下一步查看这次调用的实际运行指标。

## 3.4 查看真实运行指标
一次结果还需要对应到实际使用的 Provider、模型、Token、延迟和停止原因。下面直接读取 API 返回的 usage，并把本次探针的运行事实集中显示出来。

In [9]:
# Token 数来自真实 API 响应，不使用上一章的本地估算值
# Provider 未返回本次金额，因此成本明确记录为未知值
probe_choice = probe_response.choices[0]
probe_usage = probe_response.usage
probe_metrics = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "input_tokens": probe_usage.prompt_tokens,
    "output_tokens": probe_usage.completion_tokens,
    "total_tokens": probe_usage.total_tokens,
    "cost_usd": None,
    "latency_ms": probe_latency_ms,
    "stop_reason": probe_choice.finish_reason,
}

print(probe_metrics)

{'provider': 'openai', 'model': 'LongCat-2.0', 'input_tokens': 227, 'output_tokens': 136, 'total_tokens': 363, 'cost_usd': None, 'latency_ms': 4582, 'stop_reason': 'tool_calls'}


输出记录了本次真实请求的模型来源、输入与输出 Token、延迟和停止原因。至此已经确认共同任务、工具协议和真实 API 可以正常工作；下一章将只定义生产中常见的错误基线组件。

# 4. 定义基线组件
## 4.1 定义末尾保留装箱器
生产中常见的错误基线是每轮注入全部详细工具，再把工具占用从总预算中扣除；如果剩余空间装不下文字，就直接删除最早的 Token，只保留末尾。这种方法代码很少，却不知道被删掉的是普通历史还是当前规则。

In [10]:
def build_baseline_request():
    # 详细工具先占用共同预算，剩余空间才分给文字
    # 所有文字按原顺序拼接，超出部分直接从开头删除
    tool_tokens = count_tokens(verbose_tools)
    text_budget = TOTAL_INPUT_BUDGET - tool_tokens
    full_text = current_rules + "\n" + old_history + "\n【本次任务】" + task_request
    full_tokens = token_encoder.encode(full_text)
    visible_text = token_encoder.decode(full_tokens[-text_budget:])
    messages = [{"role": "user", "content": visible_text}]

    return {
        "tools": verbose_tools,
        "messages": messages,
        "tool_tokens": tool_tokens,
        "text_budget": text_budget,
        "visible_text": visible_text,
    }

print("基线组件：全部详细工具 + 只保留文字末尾")

基线组件：全部详细工具 + 只保留文字末尾


输出说明错误基线已经定义，但尚未组装输入或调用模型。下一章会运行这个组件，先查看模型实际能看到哪些内容，再用真实 API 生成发布计划。

# 5. 展示基线故障
## 5.1 查看基线实际输入
先运行错误基线装箱器。下面同时显示工具占用、剩余文字预算和最终可见文字，使“工具说明变长以后删掉了什么”可以直接观察。

In [11]:
# 运行第 4 章定义的同一个基线组件，不手工修改其结果
# visible_text 就是下一步真实模型能够看到的全部任务文字
baseline_request = build_baseline_request()

print("工具占用：", baseline_request["tool_tokens"], "tokens")
print("文字预算：", baseline_request["text_budget"], "tokens")
print("模型可见文字：")
print(baseline_request["visible_text"])

工具占用： 443 tokens
文字预算： 177 tokens
模型可见文字：
�记录。
【过期历史】此前多个内部服务都采用全量发布，旧流程由 ops-team 负责，错误率超过 5% 才回退。旧记录还要求先读取历史、发送群通知、等待人工回复，再保存计划。历史示例：service=catalog-api; window=22:00-22:30; strategy=all_at_once; rollback_error_rate=5%; owner=ops-team。这些记录来自已经下线的发布流程，仅用于本实验观察上下文冲突。
【本次任务】请为 catalog-api 生成今晚的发布计划，并调用 save_release_plan。


输出显示详细工具占用了 443 Token，文字只剩 177 Token。模型仍能看到过期历史和本次任务，却已经看不到开头的当前规则；下一步把这份真实可见输入发送给同一个模型。

## 5.2 获取基线真实响应
为了只观察上下文内容如何影响计划，本节固定调用 `save_release_plan`。三件详细工具仍会完整进入请求并占用预算，但模型这一步只需要填写计划正文。

In [12]:
# 请求使用第 4 章返回的消息和全部详细工具
# 指定保存工具，把实验变量集中在 plan 内容而不是工具选择
baseline_started = perf_counter()
baseline_response = client.chat.completions.create(
    model=model_name,
    messages=baseline_request["messages"],
    tools=baseline_request["tools"],
    tool_choice={
        "type": "function",
        "function": {"name": "save_release_plan"},
    },
    temperature=0,
)
baseline_latency_ms = round((perf_counter() - baseline_started) * 1000)
print("基线真实响应已收到")

基线真实响应已收到


输出说明真实模型已经根据残缺上下文返回响应。工具还没有执行，下一步只读取模型填写的计划正文，观察它采用了当前规则还是过期历史。

## 5.3 查看基线计划
模型返回的计划位于结构化工具参数中。下面读取 `plan` 字段并原样显示，不对内容做修补。

In [13]:
# 第一条 choice 保存本次模型决定，tool_calls 保存结构化动作
# arguments 是 JSON 文本，转成字典后读取原始 plan 字段
baseline_choice = baseline_response.choices[0]
baseline_call = baseline_choice.message.tool_calls[0]
baseline_arguments = json.loads(baseline_call.function.arguments)
baseline_plan = baseline_arguments["plan"]

print("工具名称：", baseline_call.function.name)
print("基线计划：", baseline_plan)

工具名称： save_release_plan
基线计划： service=catalog-api; window=22:00-22:30; strategy=all_at_once; rollback_error_rate=5%; owner=ops-team


输出中的计划来自本次真实 API。它采用了可见的过期发布方式，而不是已经被截掉的当前规则；下一步逐项与第 2 章固定的正确答案比较。

## 5.4 判断基线结果
成功计划必须包含第 2 章规定的五个“字段=值”。下面逐项显示是否命中；只要缺少或写错一项，完整任务就算失败。

In [14]:
# expected_fragments 直接由共同任务生成，不为基线放宽标准
# 显式循环逐项检查，任何错误都会把完整结果设为 False
expected_fragments = []
for field_name, expected_value in task_facts.items():
    fragment = f"{field_name}={expected_value}"
    expected_fragments.append(fragment)

baseline_success = True
for fragment in expected_fragments:
    fragment_found = fragment in baseline_plan
    print(fragment, "：", fragment_found)
    if not fragment_found:
        baseline_success = False

print("基线任务成功：", baseline_success)

service=catalog-api ： True
window=22:00-22:30 ： True
strategy=canary ： False
rollback_error_rate=2% ： False
owner=release-team ： False
基线任务成功： False


输出中的 `False` 直接指出哪些当前值没有进入计划，最终任务结果也是 `False`。模型执行了可见信息，真正出错的是 Harness 用冗长工具挤掉当前规则；最后记录这次真实调用的运行指标。

## 5.5 查看基线运行指标
下面集中显示这次基线调用的 Provider、模型、Token、延迟和停止原因，为后面的同模型修复结果保留可直接比较的记录。

In [15]:
# usage 是本次基线真实响应返回的实际 Token 统计
# Provider 未返回本次金额，因此成本继续明确记录为未知值
baseline_usage = baseline_response.usage
baseline_metrics = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "input_tokens": baseline_usage.prompt_tokens,
    "output_tokens": baseline_usage.completion_tokens,
    "total_tokens": baseline_usage.total_tokens,
    "cost_usd": None,
    "latency_ms": baseline_latency_ms,
    "stop_reason": baseline_choice.finish_reason,
}

print(baseline_metrics)

{'provider': 'openai', 'model': 'LongCat-2.0', 'input_tokens': 327, 'output_tokens': 277, 'total_tokens': 604, 'cost_usd': None, 'latency_ms': 8369, 'stop_reason': 'tool_calls'}


输出保存了错误基线的真实运行事实。到这里，已经完整复现“工具说明更详细、关键规则反而丢失、最终任务失败”的跨层回归；下一章将只定义改进组件。

# 6. 定义改进组件
## 6.1 按当前步骤选择工具
截至 2026 年 8 月，更稳妥的生产做法不是每轮暴露整个工具库，而是只提供当前步骤真正需要的能力，也称为按需暴露或渐进式披露。本次步骤只需要保存发布计划，因此先从精简工具中选出 `save_release_plan`。

In [16]:
def select_tools_for_step(tools, required_name):
    # 逐个读取工具名称，只保留当前步骤明确需要的工具
    # 返回格式仍是标准工具列表，上层 API 调用方式不变
    selected_tools = []
    for tool in tools:
        tool_name = tool["function"]["name"]
        if tool_name == required_name:
            selected_tools.append(tool)
    return selected_tools

improved_tools = select_tools_for_step(compact_tools, "save_release_plan")
print("按需工具：", improved_tools[0]["function"]["name"])

按需工具： save_release_plan


输出中只剩 `save_release_plan`。这一步没有删除系统能力，完整工具库仍保存在 `compact_tools` 中；它只是让当前请求不再为两个无关工具支付上下文成本。下一步把省出的预算优先留给当前规则。

## 6.2 定义优先级装箱器
只精简工具还不够，Harness 还必须明确哪些文字不能被历史挤掉。下面把当前规则和本次任务当作跨层契约完整保留，再用剩余预算装入过期历史；即使历史继续增长，关键输入的位置也不会被它取代。

In [17]:
def build_improved_request():
    # 当前规则和本次任务先获得预算，历史只能使用剩余空间
    # 按需工具使用精简说明，避免无关描述反复进入上下文
    tool_tokens = count_tokens(improved_tools)
    required_prefix = current_rules + "\n"
    required_suffix = "\n【本次任务】" + task_request
    required_tokens = count_tokens(required_prefix + required_suffix)
    history_budget = TOTAL_INPUT_BUDGET - tool_tokens - required_tokens
    history_tokens = token_encoder.encode(old_history)
    visible_history = token_encoder.decode(history_tokens[:history_budget])
    visible_text = required_prefix + visible_history + required_suffix
    messages = [{"role": "user", "content": visible_text}]

    return {
        "tools": improved_tools,
        "messages": messages,
        "tool_tokens": tool_tokens,
        "required_tokens": required_tokens,
        "history_budget": history_budget,
        "visible_text": visible_text,
    }

print("改进组件：先保留当前规则和任务，再装入历史")

改进组件：先保留当前规则和任务，再装入历史


输出说明优先级装箱器已经定义，但尚未组装输入或调用模型。下一章会运行这两个改进组件，查看同一总预算下的可见内容和真实发布计划。

# 7. 展示修复结果
## 7.1 查看改进版本实际输入
先运行优先级装箱器。总预算仍是 620 Token，但工具只保留当前需要的一件，当前规则和任务也会先于历史获得空间；下面直接显示各部分预算和最终可见文字。

In [18]:
# 运行第 6 章定义的改进组件，不手工补回任何文字
# visible_text 就是下一步真实模型能够看到的全部任务文字
improved_request = build_improved_request()

print("工具占用：", improved_request["tool_tokens"], "tokens")
print("规则与任务：", improved_request["required_tokens"], "tokens")
print("历史可用预算：", improved_request["history_budget"], "tokens")
print("模型可见文字：")
print(improved_request["visible_text"])

工具占用： 60 tokens
规则与任务： 91 tokens
历史可用预算： 469 tokens
模型可见文字：
【当前规则】只调用 save_release_plan。plan 必须写成一行：service=catalog-api; window=22:00-22:30; strategy=canary; rollback_error_rate=2%; owner=release-team。当前规则优先于所有历史记录。
【过期历史】此前多个内部服务都采用全量发布，旧流程由 ops-team 负责，错误率超过 5% 才回退。旧记录还要求先读取历史、发送群通知、等待人工回复，再保存计划。历史示例：service=catalog-api; window=22:00-22:30; strategy=all_at_once; rollback_error_rate=5%; owner=ops-team。这些记录来自已经下线的发布流程，仅用于本实验观察上下文冲突。
【本次任务】请为 catalog-api 生成今晚的发布计划，并调用 save_release_plan。


输出显示按需工具占用明显下降，当前规则、本次任务和历史都能进入上下文。即使新旧信息同时存在，模型也能看到“当前规则优先”的明确契约；下一步发送这份输入。

## 7.2 获取改进版本真实响应
本节继续使用同一个真实模型、零温度和指定的 `save_release_plan` 工具。除了改进后的工具列表与可见上下文，其余调用条件与基线一致。

In [19]:
# 请求只使用第 6 章组件返回的消息和按需工具
# 指定同一个保存工具，使两条路径比较同一种模型动作
improved_started = perf_counter()
improved_response = client.chat.completions.create(
    model=model_name,
    messages=improved_request["messages"],
    tools=improved_request["tools"],
    tool_choice={
        "type": "function",
        "function": {"name": "save_release_plan"},
    },
    temperature=0,
)
improved_latency_ms = round((perf_counter() - improved_started) * 1000)
print("改进版本真实响应已收到")

改进版本真实响应已收到


输出说明真实模型已经根据完整上下文返回响应。下一步读取模型填写的计划正文，确认当前规则是否真正影响了最终动作。

## 7.3 查看改进版本计划
与基线相同，发布计划保存在结构化工具参数的 `plan` 字段中。下面原样读取并显示，不对模型结果做后处理。

In [20]:
# 第一条 choice 保存改进版本的模型决定和结构化动作
# arguments 转成字典后，直接取得模型原始 plan 字段
improved_choice = improved_response.choices[0]
improved_call = improved_choice.message.tool_calls[0]
improved_arguments = json.loads(improved_call.function.arguments)
improved_plan = improved_arguments["plan"]

print("工具名称：", improved_call.function.name)
print("改进计划：", improved_plan)

工具名称： save_release_plan
改进计划： service=catalog-api; window=22:00-22:30; strategy=canary; rollback_error_rate=2%; owner=release-team


输出中的计划来自本次真实 API，并采用当前发布方式、回退阈值和负责人。下一步继续使用基线版本的五项标准逐项比较。

## 7.4 判断改进结果
为了公平比较，本节直接复用第 5 章由共同任务生成的五个“字段=值”，不为改进版本增加或放宽条件。

In [21]:
# expected_fragments 与基线版本完全相同，来自第 2 章任务
# 显式循环逐项检查，所有当前值命中才算完整成功
improved_success = True
for fragment in expected_fragments:
    fragment_found = fragment in improved_plan
    print(fragment, "：", fragment_found)
    if not fragment_found:
        improved_success = False

print("改进任务成功：", improved_success)

service=catalog-api ： True
window=22:00-22:30 ： True
strategy=canary ： True
rollback_error_rate=2% ： True
owner=release-team ： True
改进任务成功： True


输出中五项结果都应为 `True`，完整任务结果也是 `True`。模型、任务和总预算没有改变，成功来自 Harness 同时减少无关工具占用并保护当前规则；最后记录真实调用指标。

## 7.5 查看改进版本运行指标
下面记录改进调用的 Provider、模型、Token、延迟和停止原因。第 8 章将把这些数据与基线放在同一张消融对照中。

In [22]:
# usage 是本次改进版本真实响应返回的实际 Token 统计
# Provider 未返回本次金额，因此成本继续明确记录为未知值
improved_usage = improved_response.usage
improved_metrics = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "input_tokens": improved_usage.prompt_tokens,
    "output_tokens": improved_usage.completion_tokens,
    "total_tokens": improved_usage.total_tokens,
    "cost_usd": None,
    "latency_ms": improved_latency_ms,
    "stop_reason": improved_choice.finish_reason,
}

print(improved_metrics)

{'provider': 'openai', 'model': 'LongCat-2.0', 'input_tokens': 327, 'output_tokens': 242, 'total_tokens': 569, 'cost_usd': None, 'latency_ms': 6567, 'stop_reason': 'tool_calls'}


输出保存了改进版本的真实运行事实。到这里，同一模型已经在改进 Harness 下生成正确计划；下一章将汇总两条路径的成功率、Token、延迟和上下文变化。

# 8. 汇总消融对照
## 8.1 对比两条完整路径
两条路径使用同一个真实模型、同一项任务、同一个保存工具、同一份 620 Token 教学预算和同一套成功标准。下面把工具占用、当前规则是否可见、真实 API Token、延迟和最终结果放在一起。

In [23]:
# 每一行都直接读取第 5 章或第 7 章已经运行的真实结果
# 延迟是本次网络实测值，只用于记录，不代表固定性能结论
comparison_rows = [
    {
        "variant": "错误基线",
        "tools": len(baseline_request["tools"]),
        "tool_tokens": baseline_request["tool_tokens"],
        "current_rules_visible": current_rules in baseline_request["visible_text"],
        "api_tokens": baseline_metrics["total_tokens"],
        "latency_ms": baseline_metrics["latency_ms"],
        "cost_usd": baseline_metrics["cost_usd"],
        "success": baseline_success,
    },
    {
        "variant": "改进版本",
        "tools": len(improved_request["tools"]),
        "tool_tokens": improved_request["tool_tokens"],
        "current_rules_visible": current_rules in improved_request["visible_text"],
        "api_tokens": improved_metrics["total_tokens"],
        "latency_ms": improved_metrics["latency_ms"],
        "cost_usd": improved_metrics["cost_usd"],
        "success": improved_success,
    },
]

print("版本 | 工具数 | 工具Token | 当前规则可见 | API Token | 延迟 | 成本 | 任务成功")
for row in comparison_rows:
    print(
        row["variant"], "|", row["tools"], "|", row["tool_tokens"], "|",
        row["current_rules_visible"], "|", row["api_tokens"], "|",
        str(row["latency_ms"]) + " ms", "|", row["cost_usd"], "|", row["success"]
    )

版本 | 工具数 | 工具Token | 当前规则可见 | API Token | 延迟 | 成本 | 任务成功
错误基线 | 3 | 443 | False | 604 | 8369 ms | None | False
改进版本 | 1 | 60 | True | 569 | 6567 ms | None | True


对照显示，错误基线暴露三个详细工具，当前规则不可见，任务失败；改进版本只暴露当前工具并保护规则，任务成功。API Token 和延迟来自本次真实调用，金额因 Provider 未返回而保持未知；单次网络延迟会波动，稳定的机制结论是上下文状态与任务结果发生了预期变化。

## 8.2 总结跨层变化
最后只保留最关键的状态变化。它们把工具层的局部改动、上下文层的预算结果和最终任务行为连接成一条容易回看的因果链。

In [24]:
# 左侧来自错误基线，右侧来自同一次从头运行的改进版本
# 每项变化连接一个 Harness 决定与它造成的下游结果
coupling_effect = {
    "exposed_tools": f"{len(baseline_request['tools'])} -> {len(improved_request['tools'])}",
    "tool_tokens": f"{baseline_request['tool_tokens']} -> {improved_request['tool_tokens']}",
    "current_rules_visible": f"{current_rules in baseline_request['visible_text']} -> {current_rules in improved_request['visible_text']}",
    "release_strategy": "all_at_once -> canary",
    "task_success": f"{baseline_success} -> {improved_success}",
}

for state_name, change in coupling_effect.items():
    print(state_name, "：", change)

exposed_tools ： 3 -> 1
tool_tokens ： 443 -> 60
current_rules_visible ： False -> True
release_strategy ： all_at_once -> canary
task_success ： False -> True


输出说明因果链从工具层开始：按需工具使说明占用下降，当前规则因此重新进入上下文，真实模型随即从旧发布方式改为当前发布方式，任务结果由失败变为成功。模型本身没有更换，决定差异的是它外面的 Harness。

## 8.3 拓展
### nano 版省略了什么
为了保持 nano 版直白，本 Notebook 省略了多轮 Agent Loop、动态工具检索、供应商精确 Tokenizer 与价格表、KV-cache、并发请求和多次统计置信区间；这些不会改变本实验要证明的核心机制：局部组件变得更详细，不等于完整 Harness 变得更可靠。

### 延伸阅读

1. 2026, [OpenAI, Harness engineering](https://openai.com/index/harness-engineering)：仓库、反馈回路、可读性与工程约束如何共同塑造 Agent。
2. 2026, [Code as Agent Harness](https://arxiv.org/abs/2605.18747)：执行、状态和验证在代码接口处的耦合。
3. 2026, [Agent Harness Engineering: A Survey](https://openreview.net/pdf?id=3hXEPbG0dh)：ETCLOVG 七层与跨层耦合的系统化综述。